In [1]:
import torch

from datasets import load_dataset, load_from_disk, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
train_data = load_dataset("openai/gsm8k", "main", split="train")
split_idx = len(train_data) // 4

In [ ]:
print(train_data)

Dataset({
    features: ['question', 'answer', 'question_id'],
    num_rows: 7473
})


In [ ]:
question_ids = [f"q_{i+1}" for i in range(len(train_data))]
train_data = train_data.add_column("question_id", question_ids)

Flattening the indices:   0%|          | 0/7473 [00:00<?, ? examples/s]

In [ ]:
subset_1 = train_data.select(range(0, split_idx))
subset_2 = train_data.select(range(split_idx, split_idx*2))
subset_3 = train_data.select(range(split_idx*2, split_idx*3))
subset_4 = train_data.select(range(split_idx*3, len(train_data)))

In [30]:
subsets = [subset_1, subset_2, subset_3, subset_4]
for i in range(4):
    print(f"subset_{i+1} 할당량: {len(subsets[i])}개")

subset_1 할당량: 1868개
subset_2 할당량: 1868개
subset_3 할당량: 1868개
subset_4 할당량: 1869개


In [33]:
dataset_dict = DatasetDict({
    "subset_1": subset_1,
    "subset_2": subset_2,
    "subset_3": subset_3,
    "subset_4": subset_4,
})

dataset_dict.save_to_disk("gsm8k_training_subsets")

Saving the dataset (0/1 shards):   0%|          | 0/1868 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1868 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1868 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1869 [00:00<?, ? examples/s]

In [2]:
all_subsets = load_from_disk("gsm8k_training_subsets")
print(all_subsets)

DatasetDict({
    subset_1: Dataset({
        features: ['question', 'answer', 'question_id'],
        num_rows: 1868
    })
    subset_2: Dataset({
        features: ['question', 'answer', 'question_id'],
        num_rows: 1868
    })
    subset_3: Dataset({
        features: ['question', 'answer', 'question_id'],
        num_rows: 1868
    })
    subset_4: Dataset({
        features: ['question', 'answer', 'question_id'],
        num_rows: 1869
    })
})


In [6]:
# all_subsets.push_to_hub("hyunjaehyun/gsm8k_train_subsets_shuffled")